# TSE 2026 — Laboratório forense / Fase 5S — opção X

## Mecanismos físicos da curva nacional *publicada* EA20

Depois de investigar o recebimento BWEB por município, zona e tempo (S/T/V/W), examinamos agora **somente a cronologia de PUBLICAÇÃO dos resultados nacionais EA20**.

**Hipóteses verificáveis:**
1. A linearidade acumulada por número de seções (R² ≈ 0,99044) pode surgir da combinação de denominadores cumulativos e dos volumes efetivamente publicados, mesmo sem fixar cada lote?
2. Qual é a diferença entre a curva real e a curva que atribui **taxa constante de Lula a todos os votos publicados depois das 20h04**, preservando todos os volumes observados?
3. O que acontece ao prever os lotes restantes **usando somente os primeiros dez incrementos**, sem conhecer o final?
4. Os lotes efetivamente publicados tiveram uma composição uniforme ou mudaram de forma relevante ao longo da cauda?

**Distinção fundamental:** o modelo com taxa final da cauda é um **oráculo retrospectivo**: usa resultado futuro e não serve como prova de um mecanismo real. Um R² alto mede a forma da curva, não a probabilidade de fraude.

**Dados usados:** 193 snapshots EA20 nacionais do 1º turno de 2026 já versionados e com SHA. A opção X não lê BWEB, não reabre ZIP, não acessa rede nem SQLite.

## 1 — Carregar o relatório da opção X e verificar SHA

A opção X do BAT gera três arquivos locais em `H:\TSE\forensics\urnas\analysis\periodicidade_bweb_2026`. Nenhum download manual é necessário.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_repo():
    for p in (Path.cwd(),*Path.cwd().parents):
        if (p/"start_windows.bat").is_file():
            return p
    raise FileNotFoundError("Abra o notebook na pasta do repositorio TSE.")

REPO=find_repo()
sys.path.insert(0,str(REPO))
from urnas import root_path
ROOT=root_path(None)
BASE=ROOT/"analysis"/"periodicidade_bweb_2026"

def sha256(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(4*1024*1024),b""):
            h.update(block)
    return h.hexdigest()

path=BASE/"phase5s_ea20_publication_mechanics_2026_summary.json"
summary=json.loads(path.read_text(encoding="utf-8"))
assert summary["status"]=="MECANISMO_DA_CURVA_PUBLICADA_EA20_2026"
assert summary["round"]==1 and summary["year"]==2026
source=REPO/"data/forensics/analysis/phase5a_presidential_national_2026.csv"
assert sha256(source)==summary["source_sha256"]["EA20_national"]

def read_verified(meta):
    p=Path(meta["path"])
    if not p.is_file():
        p=BASE/p.name
    assert p.is_file(),f"Arquivo ausente: {p}"
    assert sha256(p)==meta["sha256"],f"SHA-256 invalido: {p.name}"
    return pd.read_csv(p,sep=";",encoding="utf-8-sig")

batches=read_verified(summary["output_files"]["batches"])
blocks=read_verified(summary["output_files"]["blocks"])
batches["data_hora"]=pd.to_datetime(batches["brt"])
assert len(batches)==152 and batches["d_valid"].sum()==18809656
assert blocks["valid"].sum()==18809656
print("OK: 193 snapshots oficiais preservados; 153 pontos pos-corte; 152 incrementos.")
print("Relatorios SHA verificados. Nenhum arquivo eleitoral foi alterado.")
display(blocks.round(4))

## 2 — Mesmo volume publicado, participação fixa nos lotes

O controle `constant_tail_oracle` preserva a quantidade **real de votos válidos e seções** de cada atualização, mas acrescenta votos de Lula em uma proporção constante igual à **taxa final da cauda**, conhecida apenas a posteriori. Isso isola a geometria cumulativa dos denominadores e da sequência de volumes.

In [ ]:
fit=summary["fits"]
table=pd.DataFrame([
    {"Eixo":"Seções publicadas","R² real":fit["sections"]["actual"]["r2"],
     "R² composição constante":fit["sections"]["constant_tail_oracle"]["r2"]},
    {"Eixo":"Votos válidos publicados","R² real":fit["valid"]["actual"]["r2"],
     "R² composição constante":fit["valid"]["constant_tail_oracle"]["r2"]},
    {"Eixo":"Relógio real EA20","R² real":fit["clock"]["actual"]["r2"],
     "R² composição constante":fit["clock"]["constant_tail_oracle"]["r2"]},
])
display(table.round(6))
fig,ax=plt.subplots(figsize=(11,5))
ax.plot(batches["cum_sections"],batches["actual_cum_lula_pct"],
        label="EA20 observado: Lula acumulado",marker=".",markersize=3)
ax.plot(batches["cum_sections"],batches["oracle_cum_lula_pct"],
        label="Composição constante (oráculo final)",linestyle="--")
ax.set(title="Linearidade no eixo de seções: observado versus contrafactual",
       xlabel="Seções publicadas acumuladas (EA20)",ylabel="Lula (% acumulado)")
ax.legend()
ax.grid(alpha=.2)
plt.show()
print("RMSE do oráculo contra a curva real:",
      summary["constant_tail_oracle"]["rmse_actual_minus_oracle_cumulative_pp"],"pp")

## 3 — Resíduos do modelo de composição constante

Mesmo um modelo que produz uma reta muito lisa pode deixar resíduos relevantes. O valor aqui é **observado menos oráculo** em pontos percentuais cumulativos, não votos irregulares. O oráculo conhece a cauda final e não é previsão independente.

In [ ]:
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(batches["cum_sections"],batches["oracle_residual_pp"])
ax.axhline(0,linestyle="--")
ax.set(title="Resíduo do EA20 ante taxa constante da cauda (circular)",
       xlabel="Seções acumuladas",ylabel="Lula real menos oráculo (pp)")
ax.grid(alpha=.2)
plt.show()
display(batches.assign(abs_residual=batches["oracle_residual_pp"].abs())
        .nlargest(12,"abs_residual")
        [["brt","d_valid","d_sections","batch_lula_pct","oracle_residual_pp"]].round(4))

## 4 — Modelo realmente pré-informado: primeiros 10 incrementos

Aqui a taxa de Lula é estimada **apenas com os dez primeiros lotes depois de 20:04:39**, e então congelada. Nos lotes restantes usamos somente os volumes posteriormente publicados para calcular a trajetória condicional. Nunca usamos os votos futuros para calibrar a taxa.

Este exercício **não prevê o número de seções nem o volume futuro de votos**; é um teste da estabilidade da composição dos lotes.

In [ ]:
early=summary["early_ten_update_training"]
display(pd.DataFrame([early]).round(5).T)
future=batches.loc[batches["early_forecast_cum_lula_pct"].notna()]
fig,ax=plt.subplots(figsize=(11,5))
ax.plot(future["cum_sections"],future["actual_cum_lula_pct"],
        label="Lula acumulado real EA20")
ax.plot(future["cum_sections"],future["early_forecast_cum_lula_pct"],
        label="Taxa calibrada nos 10 primeiros lotes",linestyle="--")
ax.set(title="Teste fora do período de treino: previsão de composição estável",
       xlabel="Seções acumuladas",ylabel="Lula (% acumulado)")
ax.legend()
ax.grid(alpha=.2)
plt.show()
print("Diferença no resultado final (previsto - real):",
      round(early["forecast_final_error_pp"],4),"pp")

## 5 — Por que a média final pode esconder composição variável?

Dividimos a cauda em quatro grupos **consecutivos e indivisíveis** de lotes, procurando cerca de 25% dos votos válidos por grupo. Nenhum lote é artificialmente dividido entre blocos. Analise as taxas, o número de atualizações e o volume.

In [ ]:
display(blocks[["block","updates","valid","d_sections","lula","flavio",
                "lula_pct","flavio_pct","first_brt","last_brt"]].round(3))
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(blocks["block"].astype(str),blocks["lula_pct"])
ax.axhline(summary["published_tail"]["lula_pct"],linestyle="--",
           label="Participação final da cauda")
ax.set(title="Composição dos lotes publicados em quatro blocos",
       xlabel="Bloco cronológico (volumes aproximadamente iguais)",
       ylabel="Lula (% dos votos novos do bloco)")
ax.set_ylim(0,100)
ax.legend()
ax.grid(axis="y",alpha=.2)
plt.show()
last=summary["last_20_updates"]
print(f"Últimas 20 atualizações: {last['valid']:,} válidos,"
      f" {last['lula_pct']:.2f}% Lula,"
      f" somente {100*last['fraction_of_tail']:.3f}% da cauda")

## 6 — Lotes incrementais: não confundir proporção do lote com acumulada

A curva **acumulada** muda lentamente quando o denominador já ultrapassou 100 milhões, mesmo que pequenos lotes tenham percentuais muito diferentes. Por isso o percentual incremental por lote é mais revelador da composição operacional.

In [ ]:
fig,ax=plt.subplots(figsize=(11,5))
larger=batches.loc[batches["d_valid"]>=5000]
ax.scatter(larger["data_hora"],larger["batch_lula_pct"],
           s=8+80*larger["d_valid"]/larger["d_valid"].max(),alpha=.6)
ax.axhline(summary["published_tail"]["lula_pct"],linestyle="--")
ax.set(title="Participação de Lula em cada atualização EA20 (> = 5 mil votos válidos)",
       xlabel="Horário REAL de publicação EA20",ylabel="Lula (%) no lote publicado")
ax.grid(alpha=.2)
fig.autofmt_xdate()
plt.show()
display(batches.nlargest(12,"d_valid")
        [["brt","d_seconds","d_sections","d_valid","batch_lula_pct","batch_flavio_pct"]].round(3))

## 7 — Qual composição seria necessária para uma reta exatamente linear?

A fórmula do controle pega a reta que liga as duas extremidades do trecho no plano `(seções acumuladas, % de Lula)` e resolve **quantos votos de Lula cada atualização precisaria acrescentar**, dados seus votos válidos e suas seções reais. É uma **restrição contábil física**, não uma alegação de que um algoritmo escolheu votos para seguir essa reta.

In [ ]:
diag=summary["endpoint_line_diagnostic"]
print("Atualizações que exigiriam proporção fora de [0%, 100%] para reta exata:",
      diag["unphysical_required_batch_shares"])
fig,ax=plt.subplots(figsize=(11,5))
valid=batches.loc[batches["d_valid"]>=5000]
ax.plot(valid["cum_sections"],valid["batch_lula_pct"],
        label="Proporção realmente publicada por lote",marker=".",markersize=3)
ax.plot(valid["cum_sections"],valid["required_batch_lula_pct_for_endpoint_line"],
        label="Proporção requerida para reta exata",alpha=.8)
ax.set(title="Taxas por lote observadas versus taxas requeridas para reta perfeita",
       xlabel="Seções publicadas acumuladas",
       ylabel="Lula (%) no lote")
ax.legend()
ax.grid(alpha=.2)
plt.show()

## 8 — Conclusão provisória e próximo teste

- Uma curva acumulada **muito linear** pode ser produzida pela própria sequência física de volumes EA20 quando se acrescentam votos em proporção constante. O modelo usado aqui é circular e deve ser rotulado como tal.
- Os lotes efetivamente publicados **não** mantiveram composição constante: sua distribuição foi se alterando, como mostram os quatro blocos e o modelo de treino inicial.
- O crescimento acumulado pode parecer regular mesmo quando as proporções incrementais mudam — isto precisa ser explicado na cadeia de totalização.
- A fase X é sobre **PUBLICAÇÃO EA20**. As fases T/V/W foram sobre **RECEBIMENTO BWEB**. Não associar diretamente seus minutos ou lotes sem dados de vínculo operacional.

**Próxima investigação:** determinar se a cronologia estadual/local efetivamente publicada reproduz o agregado nacional e se existe fila/buffering mensurável ou outra regra de agregação entre BU recebido e atualização EA20. Para isso precisaremos de snapshots estaduais sincronizados ou registros oficiais de publicação que preservem o vínculo de seções. Não inventaremos alinhamento pela hora aproximada.

In [ ]:
print("Fase:",summary["version"])
print("EA20 nacional SHA-256:",summary["source_sha256"]["EA20_national"])
print("Corte:",summary["cutoff_brt"])
print("Publicados após o corte:",summary["published_tail"]["valid"],"válidos")
print("R² observado por seções:",summary["fits"]["sections"]["actual"]["r2"])
print("R² taxa constante no oráculo:",summary["fits"]["sections"]["constant_tail_oracle"]["r2"])
print("Todos os dados originais preservados.")